In [4]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

# PDF

In [5]:
# load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob = "**/*.pdf",
    loader_cls = PyMuPDFLoader,
    show_progress=True
)

In [6]:
pdf_documents = dir_loader.load()

for pdf in pdf_documents:
    print(pdf)
    print()

100%|██████████| 2/2 [00:01<00:00,  1.87it/s]

page_content='CS8591 – Computer Networks                                                      Unit 3
5

When a router receives a packet while processing another packet, the received 
packet needs to be stored in the input buffer waiting for its turn.

A router has an input buffer with a limited size. 

A time may come when the buffer is full and the next packet needs to be 
dropped. 

The effect of packet loss on the Internet network layer is that the packet needs 
to be resent, which in turn may create overflow and cause more packet loss.
CONGESTION CONTROL

Congestion at the network layer is related to two issues, throughput and delay.
Based on Delay

When the load is much less than the capacity of the network, the delay is at a 
minimum.

This minimum delay is composed of propagation delay and processing delay, 
both of which are negligible. 

However, when the load reaches the network capacity, the delay increases 
sharply because we now need to add the queuing delay to the

In [7]:
print(type(pdf_documents))
print(type(pdf_documents[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [8]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [9]:
chunks=split_documents(pdf_documents)

Split 39 documents into 84 chunks

Example chunk:
Content: CS8591 – Computer Networks                                                      Unit 3
5

When a router receives a packet while processing another packet, the received 
packet needs to be stored in t...
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260513154605', 'source': '..\\data\\pdf\\congestion-control.pdf', 'file_path': '..\\data\\pdf\\congestion-control.pdf', 'total_pages': 4, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260513154605', 'page': 0}


### embedding and vectorStoreDB

#### Embedding

In [10]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__ (self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings.
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Errror loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings from a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3077.61it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\agarw\AppData\Local\Temp\ipykernel_10196\3024836269.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### vectorStoreDB

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents
            embeddings: Correspoding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding 
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store!")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
            

In [13]:
vectorStore = VectorStore()
vectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 84


In [14]:
chunks

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260513154605', 'source': '..\\data\\pdf\\congestion-control.pdf', 'file_path': '..\\data\\pdf\\congestion-control.pdf', 'total_pages': 4, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260513154605', 'page': 0}, page_content='CS8591 – Computer Networks                                                      Unit 3\n5\n\uf0b7\nWhen a router receives a packet while processing another packet, the received \npacket needs to be stored in the input buffer waiting for its turn.\n\uf0b7\nA router has an input buffer with a limited size. \n\uf0b7\nA time may come when the buffer is full and the next packet needs to be \ndropped. \n\uf0b7\nThe effect of packet loss on the Internet network layer is that the packet needs \nto be resent, which in turn may create overflow and cause more packet loss.\nCONGESTION CONTROL\n\uf

In [15]:
# convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store in the vector database
vectorStore.add_documents(chunks, embeddings)

Generating embeddings for 84 texts...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.30it/s]

Generated embeddings with shape: (84, 384)
Adding 84 documents to vector store...
Successfully added 84 documents to vector store!
Total documents in collection: 168


# Retriever Pipeline from VectorStore

In [16]:
from importlib.metadata import metadata
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve (self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]: 
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score_threshold: {score_threshold}")

        # Generate query embeddings
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print ("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 


In [17]:
rag_retriever = RAGRetriever(vectorStore, embedding_manager)
rag_retriever

In [18]:
rag_retriever.retrieve("What is a router?")

Retrieving documents for query: 'What is a router?'
Top K: 5, Score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5f6df742_22',
  'content': 'Figure Difference between a switch and a bridge \n \n  \n5. Routers – A router is a device like a switch that routes data packets based on \ntheir IP addresses. Router is mainly a Network Layer device. Routers normally \nconnect LANs and WANs together and have a dynamically updating routing \ntable based on which they make decisions on routing the data packets. Router \ndivide broadcast domains of hosts connected through it. \n \nA router is used to route data packets between two networks. It reads the information in \neach packet to tell where it is going. If it is destined for an immediate network it has \naccess to, it will strip the outer packet (IP packet for example), readdress the packet to \nthe proper ethernet address, and transmit it on that network. If it is destined for another \nnetwork and must be sent to another router, it will re-package the outer packet to be \nreceived by the next router and send it to the next router. Routing 

# Integration of Vectordb context pipeline with LLM output

In [49]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables from .env file
load_dotenv()

# Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")
groq_llm = ChatGroq(groq_api_key=groq_api_key, model_name = "llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question"

    # generate the answer using groq llm
    prompt = f"""Use the following context to answer the question concisely.
                Context:
                {context}
                
                Question: {query}
                
                Answer:
                
                Format: Give the answer in bullet points"""

    response = groq_llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [50]:
answer = rag_simple("What is the difference between a switch and a rounter?", rag_retriever, groq_llm)
print(answer)

Retrieving documents for query: 'What is the difference between a switch and a rounter?'
Top K: 3, Score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 97.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


* A switch is a data link layer device, whereas a router is a network layer device.
* A switch divides the collision domain of hosts, but the broadcast domain remains the same, whereas a router divides both the collision and broadcast domains.
* A switch forwards packets based on MAC addresses, whereas a router forwards packets based on IP addresses.
* A switch maintains a directory of MAC addresses and port numbers, whereas a router maintains a routing table of IP addresses and network paths.
* A switch is primarily used for LAN connectivity, whereas a router is used for both LAN and WAN connectivity.


# Enhanced RAG Pipeline Features

In [51]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, scores, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold = min_score)

    if not results:
        return {'answer': 'No relevant context found.', 'sources':[], 'confidence': 0.0, 'context': ''}

    # prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [
        {
            'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
            'page': doc['metadata'].get('page', 'unknown'),
            'score': doc['similarity_score'],
            'preview': doc['content'][:120] + '...'
        } for doc in results
    ]
    confidence = max([doc['similarity_score'] for doc in results])

    # generate answer
    prompt = f"""Use the following context to answer the question concisely.
                Context:
                {context}
                
                Question: {query}
                
                Answer:
                
                Format: Give the answer in bullet points"""

    response = groq_llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer' : response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output

In [52]:
# Example usuage

result = rag_advanced("What is meant by congestion control?", rag_retriever, groq_llm, top_k=3, min_score=0.1, return_context=True)

Retrieving documents for query: 'What is meant by congestion control?'
Top K: 3, Score_threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


In [54]:
print("Answer: ", result['answer'])
print("Sources: ", result['sources'])
print("Confidence: ", result['confidence'])
print("Context Preview: ", result['context'][:300])

Answer:  * Congestion control is a mechanism for improving performance.
* It refers to techniques and mechanisms that can either prevent congestion before it happens or remove congestion after it has happened.
Sources:  [{'source': '..\\data\\pdf\\congestion-control.pdf', 'page': 0, 'score': 0.48278069496154785, 'preview': 'the packets do not reach the destinations.\n   Congestion Control Mechanisms\n\uf0b7\nCongestion control is a mechanism for impr...'}, {'source': '..\\data\\pdf\\congestion-control.pdf', 'page': 0, 'score': 0.48278069496154785, 'preview': 'the packets do not reach the destinations.\n   Congestion Control Mechanisms\n\uf0b7\nCongestion control is a mechanism for impr...'}]
Confidence:  0.48278069496154785
Context Preview:  the packets do not reach the destinations.
   Congestion Control Mechanisms

Congestion control is a mechanism for improving performance. 

It refers to techniques and mechanisms that can either prevent congestion 
before it happens or remove con

In [57]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [58]:
adv_rag = AdvancedRAGPipeline(rag_retriever, groq_llm)
result = adv_rag.query("what is the difference between a switch and a bridge?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is the difference between a switch and a bridge?'
Top K: 3, Score_threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Figure Difference between a switch and a bridge 
 
  
5. Routers – A router is a device like a switch that routes data packets based on 
their IP addresses. Router is mainly a Network Layer device. Routers normally 
connect LANs and WANs together and 

have a dynamically updating routing 
table based on which they make decisions on routing the data packets. Router 
divide broadcast domains of hosts connected through it. 
 
A router is used to route data packets between two networks. It reads the information in 
each packet to tell where it is going. If it is destined for an immediate network it has 
access to, it will strip the outer packet (IP packet for example), readdress the packet to 
the proper ethernet address, and transmit it on that network. If it is destined for another 
network and must be sent to another router, it will re-package the outer packet to be 
received by the next router and send it to the next router. Routing occurs at the network

Figure Difference between a switch and a bridge 
 
  
5. Routers – A router is a device like a switch that routes data packets based on 
their IP addresses. Router is mainly a Network Layer device. Routers normally 
connect LANs and WANs together and have a dynamically updating rout